In [1]:
print("hello")

hello


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

print("All libraries imported successfully!")

All libraries imported successfully!


In [3]:
df = pd.read_csv("../data/processed_news_nlp.csv")

print("Dataset shape:", df.shape)
print(df[["processed_text", "label"]].head())

Dataset shape: (38646, 10)
                                      processed_text  label
0  ben stein call th circuit court committed coup...      0
1  trump drop steve bannon national security coun...      1
2  puerto rico expects u lift jones act shipping ...      1
3  oops trump accidentally confirmed leaked israe...      0
4  donald trump head scotland reopen golf resort ...      1


In [6]:
# Remove missing values from processed text
df["processed_text"] = df["processed_text"].fillna("").astype(str)

X = df["processed_text"]
y = df["label"]

print("Missing processed text:", df["processed_text"].isna().sum())
print("Dataset shape:", df.shape)

Missing processed text: 0
Dataset shape: (38646, 10)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training articles:", len(X_train))
print("Testing articles:", len(X_test))

Training articles: 30916
Testing articles: 7730


In [8]:
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (30916, 50000)
Testing TF-IDF shape: (7730, 50000)


In [9]:
log_reg = LogisticRegression(max_iter=1000)

log_reg.fit(X_train_tfidf, y_train)

y_pred_lr = log_reg.predict(X_test_tfidf)

print("Logistic Regression Accuracy:",
      accuracy_score(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.9897800776196637


In [10]:
nb = MultinomialNB()

nb.fit(X_train_tfidf, y_train)

y_pred_nb = nb.predict(X_test_tfidf)

print("Naive Bayes Accuracy:",
      accuracy_score(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.9575679172056921


In [11]:
svm = LinearSVC()

svm.fit(X_train_tfidf, y_train)

y_pred_svm = svm.predict(X_test_tfidf)

print("Linear SVM Accuracy:",
      accuracy_score(y_test, y_pred_svm))

Linear SVM Accuracy: 0.9961190168175937


In [12]:
models = {
    "Logistic Regression": y_pred_lr,
    "Naive Bayes": y_pred_nb,
    "Linear SVM": y_pred_svm
}

for name, predictions in models.items():
    print(f"\n{name}")
    print("-" * 40)
    print("Accuracy :", accuracy_score(y_test, predictions))
    print("Precision:", precision_score(y_test, predictions))
    print("Recall   :", recall_score(y_test, predictions))
    print("F1 Score :", f1_score(y_test, predictions))


Logistic Regression
----------------------------------------
Accuracy : 0.9897800776196637
Precision: 0.985981308411215
Recall   : 0.995517810804435
F1 Score : 0.9907266111045897

Naive Bayes
----------------------------------------
Accuracy : 0.9575679172056921
Precision: 0.9619655090952044
Recall   : 0.9606039160179287
F1 Score : 0.9612842304060434

Linear SVM
----------------------------------------
Accuracy : 0.9961190168175937
Precision: 0.9943622269203665
Recall   : 0.9985845718329794
F1 Score : 0.9964689265536724


In [13]:
import joblib

joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")

print("TF-IDF vectorizer saved successfully!")

TF-IDF vectorizer saved successfully!


In [14]:
joblib.dump(svm, "../models/linear_svm_model.pkl")

print("Linear SVM model saved successfully!")

Linear SVM model saved successfully!


In [15]:
import joblib

loaded_tfidf = joblib.load("../models/tfidf_vectorizer.pkl")
loaded_svm = joblib.load("../models/linear_svm_model.pkl")

print("TF-IDF vectorizer loaded successfully!")
print("Linear SVM model loaded successfully!")

TF-IDF vectorizer loaded successfully!
Linear SVM model loaded successfully!


In [24]:
import re

In [25]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-zA-Z\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [30]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

print("Stop words loaded:", len(stop_words))

Stop words loaded: 198


In [26]:
def remove_stopwords(text):
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

In [32]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

print("Lemmatizer loaded successfully!")

Lemmatizer loaded successfully!


In [27]:
def lemmatize_text(text):
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(words)

In [28]:
def predict_news(article):
    # Clean the new article
    cleaned = clean_text(article)

    # Remove stopwords
    cleaned = remove_stopwords(cleaned)

    # Lemmatize
    cleaned = lemmatize_text(cleaned)

    # Convert to TF-IDF
    article_tfidf = loaded_tfidf.transform([cleaned])

    # Predict
    prediction = loaded_svm.predict(article_tfidf)[0]

    if prediction == 0:
        return "FAKE NEWS ❌"
    else:
        return "REAL NEWS ✅"

In [33]:
test_article = """
The government announced a new education policy that will provide
free digital learning resources to students across the country.
"""

result = predict_news(test_article)

print("Prediction:", result)

Prediction: FAKE NEWS ❌


In [34]:
test_article_2 = """
The Ministry of Education announced today that several government schools
will receive new digital classrooms and improved internet facilities.
Officials said the program will be implemented in phases across different states.
"""

result = predict_news(test_article_2)

print("Article:")
print(test_article_2)
print("\nPrediction:", result)

Article:

The Ministry of Education announced today that several government schools
will receive new digital classrooms and improved internet facilities.
Officials said the program will be implemented in phases across different states.


Prediction: REAL NEWS ✅


In [35]:
test_article_3 = """
BREAKING!!! A shocking secret has been revealed! Scientists have discovered
that drinking one glass of magical water every morning can make people
live for 200 years. The government is hiding this incredible discovery
from the public!!!
"""

result = predict_news(test_article_3)

print("Article:")
print(test_article_3)
print("\nPrediction:", result)

Article:

BREAKING!!! A shocking secret has been revealed! Scientists have discovered
that drinking one glass of magical water every morning can make people
live for 200 years. The government is hiding this incredible discovery
from the public!!!


Prediction: FAKE NEWS ❌


In [36]:
test_article_4 = """BJP TV face shehzad poonawad doubles down on resignation,cites finances
"""

result = predict_news(test_article_4)

print("Article:")
print(test_article_4)
print("\nPrediction:", result)

Article:
BJP TV face shehzad poonawad doubles down on resignation,cites finances


Prediction: FAKE NEWS ❌


In [37]:
test_article_5 = """Shehzad Poonawala, who quit Bharatiya Janata Party (BJP) last month, doubled down on his resignation in a fresh letter to the party chief, citing “pressing emergency financial and personal circumstances
"""

result = predict_news(test_article_5)

print("Article:")
print(test_article_5)
print("\nPrediction:", result)

Article:
Shehzad Poonawala, who quit Bharatiya Janata Party (BJP) last month, doubled down on his resignation in a fresh letter to the party chief, citing “pressing emergency financial and personal circumstances


Prediction: REAL NEWS ✅
